In [ ]:

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. CONFIGURATION
# ============================================================

TICKER = "SPY"
START_DATE = "2018-01-01"
END_DATE = "2024-01-01"

# Chronological train/test split
SPLIT_DATE = "2022-01-01"

# Original baseline strategy
BASELINE_LOOKBACK = 60

# Sensitivity analysis: 20 through 252 trading days
LOOKBACK_WINDOWS = range(20, 253)

TRADING_DAYS = 252

# Transaction costs per one-way trade:
# 5 bps slippage/spread + 5 bps commission/market impact
SLIPPAGE_BPS = 5
OTHER_COST_BPS = 5
TRANSACTION_COST = (SLIPPAGE_BPS + OTHER_COST_BPS) / 10000

# Risk-free rate assumed to be zero for comparability
RISK_FREE_RATE = 0.0

# ============================================================
# 2. DOWNLOAD AND PREPARE DATA
# ============================================================

raw = yf.download(
    TICKER,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

# Handle yfinance MultiIndex columns
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].iloc[:, 0]
else:
    prices = raw["Close"]

data = pd.DataFrame({"Close": prices}).dropna()

if data.empty:
    raise ValueError("No price data downloaded.")

data["return"] = data["Close"].pct_change()

# All lookbacks will be evaluated over the same dates.
# This prevents shorter lookbacks from unfairly receiving
# additional historical observations.
common_start = data.index[0] + pd.Timedelta(days=0)

# ============================================================
# 3. PERFORMANCE METRICS
# ============================================================

def sharpe_ratio(returns, periods=252, risk_free_rate=0.0):
    """
    Annualized Sharpe ratio.

    Uses zero risk-free rate by default.
    """
    returns = pd.Series(returns).dropna()

    if len(returns) < 2:
        return np.nan

    volatility = returns.std()

    if volatility == 0 or np.isnan(volatility):
        return np.nan

    daily_rf = (1 + risk_free_rate) ** (1 / periods) - 1
    excess_returns = returns - daily_rf

    return (
        excess_returns.mean() / volatility
    ) * np.sqrt(periods)


def max_drawdown(returns):
    """
    Maximum peak-to-trough loss.
    """
    returns = pd.Series(returns).dropna()

    if returns.empty:
        return np.nan

    cumulative = (1 + returns).cumprod()

    # Include initial equity of 1.0
    cumulative = pd.concat([
        pd.Series([1.0]),
        cumulative.reset_index(drop=True)
    ], ignore_index=True)

    rolling_max = cumulative.cummax()

    drawdown = cumulative / rolling_max - 1

    return drawdown.min()


def annualized_return(returns):
    """
    Annualized compounded return (CAGR).
    """
    returns = pd.Series(returns).dropna()

    if returns.empty:
        return np.nan

    total_growth = (1 + returns).prod()

    if total_growth <= 0:
        return np.nan

    return total_growth ** (
        TRADING_DAYS / len(returns)
    ) - 1


def performance_summary(returns):
    returns = pd.Series(returns).dropna()

    return {
        "Annual Return": annualized_return(returns),
        "Annual Volatility": returns.std() * np.sqrt(TRADING_DAYS),
        "Sharpe": sharpe_ratio(
            returns,
            risk_free_rate=RISK_FREE_RATE
        ),
        "Max Drawdown": max_drawdown(returns)
    }


# ============================================================
# 4. MOMENTUM STRATEGY BACKTEST
# ============================================================

def run_backtest(prices_data, lookback):
    """
    Long-only time-series momentum strategy.

    Buy SPY when trailing momentum is positive.
    Otherwise hold cash.

    Signal is delayed by one trading day to avoid
    lookahead bias.

    Transaction costs are proportional to changes
    in portfolio position.
    """

    df = prices_data.copy()

    # Trailing lookback momentum
    df["momentum"] = df["Close"].pct_change(lookback)

    # Yesterday's momentum determines today's position
    df["position"] = (
        df["momentum"].shift(1) > 0
    ).astype(float)

    # Gross strategy returns
    df["gross_return"] = (
        df["position"] * df["return"]
    )

    # Turnover counts entry and exit transactions
    df["position_change"] = (
        df["position"].diff().abs()
    )

    df["position_change"] = df["position_change"].fillna(
        df["position"].abs()
    )

    # Explicit transaction cost
    df["cost"] = (
        df["position_change"] * TRANSACTION_COST
    )

    # Net strategy returns
    df["net_return"] = (
        df["gross_return"] - df["cost"]
    )

    return df


# ============================================================
# 5. COMMON EVALUATION SAMPLE
# ============================================================

# The largest lookback needs 252 days plus a one-day signal lag.
max_lookback = max(LOOKBACK_WINDOWS)

# The first valid trading day for the largest lookback.
evaluation_start = data.index[max_lookback + 1]

print(f"Common evaluation starts: {evaluation_start.date()}")
print(f"Train/Test split: {SPLIT_DATE}")
print(f"One-way cost: {TRANSACTION_COST * 10000:.1f} bps")

# ============================================================
# 6. PARAMETER SENSITIVITY ANALYSIS
# ============================================================

results = []
backtests = {}

for lookback in LOOKBACK_WINDOWS:

    bt = run_backtest(data, lookback)

    # Evaluate all parameters over identical dates
    bt = bt.loc[bt.index >= evaluation_start].copy()

    train = bt.loc[bt.index < SPLIT_DATE]
    test = bt.loc[bt.index >= SPLIT_DATE]

    if len(train) < 2 or len(test) < 2:
        raise ValueError("Insufficient training or testing data.")

    # Calculate Sharpe before and after costs
    train_gross = sharpe_ratio(train["gross_return"])
    train_net = sharpe_ratio(train["net_return"])

    test_gross = sharpe_ratio(test["gross_return"])
    test_net = sharpe_ratio(test["net_return"])

    results.append({
        "Lookback": lookback,

        "IS Gross Sharpe": train_gross,
        "IS Net Sharpe": train_net,

        "OOS Gross Sharpe": test_gross,
        "OOS Net Sharpe": test_net,

        "IS Trades": train["position_change"].sum(),
        "OOS Trades": test["position_change"].sum(),

        "IS Total Costs": train["cost"].sum(),
        "OOS Total Costs": test["cost"].sum()
    })

    backtests[lookback] = bt


sensitivity = pd.DataFrame(results).set_index("Lookback")

# ============================================================
# 7. IN-SAMPLE OPTIMIZATION
# ============================================================

best_lookback = sensitivity["IS Net Sharpe"].idxmax()

baseline = sensitivity.loc[BASELINE_LOOKBACK]
optimized = sensitivity.loc[best_lookback]

print("\n" + "=" * 60)
print("PARAMETER OPTIMIZATION")
print("=" * 60)

print(f"Original lookback: {BASELINE_LOOKBACK} days")
print(f"Optimized lookback (IS net Sharpe): {best_lookback} days")

print("\nOriginal 60-Day Strategy:")
print(f"  IS Gross Sharpe:  {baseline['IS Gross Sharpe']:.3f}")
print(f"  IS Net Sharpe:    {baseline['IS Net Sharpe']:.3f}")
print(f"  OOS Gross Sharpe: {baseline['OOS Gross Sharpe']:.3f}")
print(f"  OOS Net Sharpe:   {baseline['OOS Net Sharpe']:.3f}")

print("\nOptimized Strategy:")
print(f"  IS Gross Sharpe:  {optimized['IS Gross Sharpe']:.3f}")
print(f"  IS Net Sharpe:    {optimized['IS Net Sharpe']:.3f}")
print(f"  OOS Gross Sharpe: {optimized['OOS Gross Sharpe']:.3f}")
print(f"  OOS Net Sharpe:   {optimized['OOS Net Sharpe']:.3f}")


# ============================================================
# 8. EXPLICIT TRANSACTION COST IMPACT
# ============================================================

print("\n" + "=" * 60)
print("TRANSACTION COST IMPACT")
print("=" * 60)

for label, row in [
    ("60-Day Baseline", baseline),
    ("IS-Optimized", optimized)
]:
    print(f"\n{label}")

    for period in ["IS", "OOS"]:

        gross = row[f"{period} Gross Sharpe"]
        net = row[f"{period} Net Sharpe"]

        print(
            f"  {period} Sharpe: "
            f"{gross:.3f} before costs "
            f"vs {net:.3f} after costs "
            f"(difference: {gross - net:+.3f})"
        )

        print(
            f"  {period} Trades: "
            f"{row[f'{period} Trades']:.0f}"
        )

# ============================================================
# 9. OVERFITTING / ROBUSTNESS DIAGNOSTICS
# ============================================================

print("\n" + "=" * 60)
print("OUT-OF-SAMPLE ROBUSTNESS")
print("=" * 60)

is_sharpe = optimized["IS Net Sharpe"]
oos_sharpe = optimized["OOS Net Sharpe"]

sharpe_decay = is_sharpe - oos_sharpe

print(f"Optimized IS Net Sharpe:  {is_sharpe:.3f}")
print(f"Optimized OOS Net Sharpe: {oos_sharpe:.3f}")
print(f"IS-to-OOS Sharpe change:  {sharpe_decay:+.3f}")

if oos_sharpe < is_sharpe:
    print(
        "Warning: OOS performance is lower than IS performance. "
        "This may indicate overfitting or a market regime change."
    )
else:
    print(
        "OOS Sharpe is at least as high as IS Sharpe. "
        "This is encouraging, but does not establish robustness."
    )

# Check whether the best result is a narrow parameter spike.
neighbor_start = max(20, best_lookback - 10)
neighbor_end = min(252, best_lookback + 10)

neighboring = sensitivity.loc[
    neighbor_start:neighbor_end,
    "IS Net Sharpe"
]

print(
    f"\nMean IS Net Sharpe in +/-10-day neighborhood: "
    f"{neighboring.mean():.3f}"
)

print(
    f"Best IS Net Sharpe: {is_sharpe:.3f}"
)

print(
    "\nInterpretation: Broad regions of similar performance "
    "are generally more reassuring than isolated peaks."
)

# ============================================================
# 10. DETAILED PERFORMANCE METRICS
# ============================================================

selected_bt = backtests[best_lookback]

selected_train = selected_bt.loc[
    selected_bt.index < SPLIT_DATE
]

selected_test = selected_bt.loc[
    selected_bt.index >= SPLIT_DATE
]

print("\n" + "=" * 60)
print("OPTIMIZED STRATEGY PERFORMANCE SUMMARY")
print("=" * 60)

for label, subset in [
    ("IN-SAMPLE", selected_train),
    ("OUT-OF-SAMPLE", selected_test)
]:

    gross_stats = performance_summary(
        subset["gross_return"]
    )

    net_stats = performance_summary(
        subset["net_return"]
    )

    print(f"\n{label}")
    print(
        f"Annual Return (Gross): "
        f"{gross_stats['Annual Return']:.2%}"
    )
    print(
        f"Annual Return (Net):   "
        f"{net_stats['Annual Return']:.2%}"
    )
    print(
        f"Annual Volatility:     "
        f"{net_stats['Annual Volatility']:.2%}"
    )
    print(
        f"Sharpe Before Costs:   "
        f"{gross_stats['Sharpe']:.3f}"
    )
    print(
        f"Sharpe After Costs:    "
        f"{net_stats['Sharpe']:.3f}"
    )
    print(
        f"Maximum Drawdown:      "
        f"{net_stats['Max Drawdown']:.2%}"
    )


# ============================================================
# 11. VISUALIZATION
# ============================================================

BG = "#0D1B2A"
CYAN = "#00D4FF"
GREEN = "#00E5A0"
ORANGE = "#FFB347"
RED = "#FF4D6D"
WHITE = "#FFFFFF"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "axes.labelcolor": WHITE,
    "text.color": WHITE,
    "xtick.color": WHITE,
    "ytick.color": WHITE,
    "font.size": 10
})

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

fig.suptitle(
    f"{TICKER} Momentum Strategy: Sensitivity and Robustness",
    fontsize=16,
    fontweight="bold",
    color=WHITE
)

for ax in axes.flat:
    ax.set_facecolor(BG)
    ax.tick_params(colors=WHITE)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["bottom"].set_color("#4a5a6a")
    ax.spines["left"].set_color("#4a5a6a")

    ax.grid(alpha=0.12)

# ------------------------------------------------------------
# Plot 1: Equity curves
# ------------------------------------------------------------

ax1 = axes[0, 0]

baseline_bt = backtests[BASELINE_LOOKBACK]

baseline_equity = (
    1 + baseline_bt["net_return"]
).cumprod()

optimized_equity = (
    1 + selected_bt["net_return"]
).cumprod()

buy_hold_equity = (
    1 + selected_bt["return"]
).cumprod()

ax1.plot(
    baseline_equity.index,
    baseline_equity,
    color=CYAN,
    label="60-Day Baseline (Net)"
)

ax1.plot(
    optimized_equity.index,
    optimized_equity,
    color=GREEN,
    label=f"IS-Optimized {best_lookback}-Day (Net)"
)

ax1.plot(
    buy_hold_equity.index,
    buy_hold_equity,
    color=WHITE,
    alpha=0.65,
    label="Buy and Hold"
)

ax1.axvline(
    pd.Timestamp(SPLIT_DATE),
    color=ORANGE,
    linestyle="--",
    label="OOS Start"
)

ax1.set_title(
    "Equity Curves (Common Sample)",
    fontweight="bold"
)

ax1.set_ylabel("Growth of $1")
ax1.legend(facecolor="#1a2a3a", labelcolor=WHITE)

# ------------------------------------------------------------
# Plot 2: Drawdown
# ------------------------------------------------------------

ax2 = axes[0, 1]

dd = (
    optimized_equity / optimized_equity.cummax()
) - 1

ax2.plot(
    dd.index,
    dd,
    color=RED
)

ax2.fill_between(
    dd.index,
    dd.values,
    0,
    alpha=0.25,
    color=RED
)

ax2.axvline(
    pd.Timestamp(SPLIT_DATE),
    color=ORANGE,
    linestyle="--"
)

ax2.set_title(
    "Optimized Strategy Drawdown (Net)",
    fontweight="bold"
)

ax2.set_ylabel("Drawdown")

# ------------------------------------------------------------
# Plot 3: Parameter sensitivity
# ------------------------------------------------------------

ax3 = axes[1, 0]

ax3.plot(
    sensitivity.index,
    sensitivity["IS Net Sharpe"],
    color=CYAN,
    label="In-Sample Net Sharpe"
)

ax3.plot(
    sensitivity.index,
    sensitivity["OOS Net Sharpe"],
    color=ORANGE,
    alpha=0.85,
    label="Out-of-Sample Net Sharpe"
)

ax3.axvline(
    best_lookback,
    color=GREEN,
    linestyle="--",
    label=f"IS Optimum = {best_lookback}"
)

ax3.axvline(
    BASELINE_LOOKBACK,
    color=WHITE,
    linestyle=":",
    alpha=0.75,
    label="Original 60 Days"
)

ax3.axhline(
    0,
    color=WHITE,
    alpha=0.3
)

ax3.set_title(
    "Lookback Sensitivity: 20-252 Days",
    fontweight="bold"
)

ax3.set_xlabel("Momentum Lookback (Trading Days)")
ax3.set_ylabel("Annualized Net Sharpe")
ax3.legend(facecolor="#1a2a3a", labelcolor=WHITE)

# ------------------------------------------------------------
# Plot 4: Gross vs Net Sharpe
# ------------------------------------------------------------

ax4 = axes[1, 1]

categories = [
    "Baseline IS",
    "Baseline OOS",
    "Optimized IS",
    "Optimized OOS"
]

gross_values = [
    baseline["IS Gross Sharpe"],
    baseline["OOS Gross Sharpe"],
    optimized["IS Gross Sharpe"],
    optimized["OOS Gross Sharpe"]
]

net_values = [
    baseline["IS Net Sharpe"],
    baseline["OOS Net Sharpe"],
    optimized["IS Net Sharpe"],
    optimized["OOS Net Sharpe"]
]

x = np.arange(len(categories))
width = 0.35

ax4.bar(
    x - width / 2,
    gross_values,
    width,
    color=CYAN,
    label="Before Costs"
)

ax4.bar(
    x + width / 2,
    net_values,
    width,
    color=ORANGE,
    label="After Costs"
)

ax4.set_xticks(x)
ax4.set_xticklabels(categories, rotation=15)

ax4.set_ylabel("Annualized Sharpe")

ax4.set_title(
    "Transaction Cost Impact on Sharpe",
    fontweight="bold"
)

ax4.axhline(0, color=WHITE, alpha=0.35)

ax4.legend(facecolor="#1a2a3a", labelcolor=WHITE)

plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.savefig(
    "momentum_robustness_analysis.png",
    dpi=150,
    bbox_inches="tight",
    facecolor=BG
)

plt.show()

# ============================================================
# 12. EXPORT FULL SENSITIVITY RESULTS
# ============================================================

sensitivity.to_csv(
    "momentum_sensitivity_results.csv"
)

print("\nAnalysis complete.")
print("Saved momentum_robustness_analysis.png")
print("Saved momentum_sensitivity_results.csv")
